# 01 — Bases de Python

Révision CPGE (Sup → Spé), filières MPSI / PCSI / PSI.

Chaque notion suit le schéma : **définition → quand l'utiliser → code testé (`assert`) → pièges → astuce**.

> Exécute les cellules avec `Shift+Entrée`. Si un `assert` ne lève rien, le comportement est conforme.


## 1.1 Variables, types et affectation

**Définition.** Une variable Python est une *référence* vers un objet ; les types de base sont `int`, `float`, `bool` et `str`.

**Quand l'utiliser.** Pour stocker des valeurs intermédiaires et structurer un calcul. Éviter de changer le type d'une même variable au milieu d'un calcul mathématique.


In [ ]:
# Affectation et typage dynamique
a = 10           # int
x = 3.14         # float
is_valid = True  # bool
nom = "CPGE"     # str

assert isinstance(a, int)
assert isinstance(x, float)
assert isinstance(is_valid, bool)
assert isinstance(nom, str)
assert a == 10

# int en précision arbitraire : aucun dépassement
assert 2**200 == 1606938044258990275541962092341162602522202993782792835301376
print("types :", type(a), type(x), type(is_valid), type(nom))


**Pièges.** `=` (affectation) ≠ `==` (test d'égalité) ; `True` / `False` prennent une majuscule.

**Astuce.** Les `int` Python sont en précision arbitraire : `2**200` est exact.


## 1.2 Boucles `for` et `while`

**Définition.** `for` parcourt une séquence dont la taille est connue ; `while` répète un bloc tant qu'une condition reste vraie.

**Quand l'utiliser.** `for` si le nombre de tours est connu à l'avance ; `while` si l'arrêt dépend d'un calcul (seuil, convergence).


In [ ]:
# for : nombre de tours connu
somme = 0
for i in range(1, 5):        # 1, 2, 3, 4
    somme += i
assert somme == 10

# while logarithmique : combien de tours pour dépasser 16 ?
i, tours = 1, 0
while i <= 16:
    i *= 2
    tours += 1
assert i == 32 and tours == 5   # 1 → 2 → 4 → 8 → 16 → 32

# while + break : sortie anticipée
n = 0
while True:
    n += 1
    if n == 3:
        break
assert n == 3


**Pièges.** Oublier de mettre à jour la variable de contrôle (boucle infinie) ; `range(1, 5)` s'arrête à **4**, pas 5.

**Astuce.** Pour un `while`, méthode en 4 étapes : variable de contrôle → valeur u_k → inéquation d'arrêt → nombre de tours.


## 1.3 Fonctions et portée (scope)

**Définition.** Une fonction regroupe un bloc d'instructions réutilisable, avec des paramètres et une valeur de retour.

**Quand l'utiliser.** Pour modulariser et éviter la répétition. Éviter de modifier une variable globale sans la passer en paramètre.


In [ ]:
def carre(x):
    res = x * x      # variable locale
    return res

def oubli(x):
    res = x * x      # pas de return → None

assert carre(5) == 25
assert oubli(5) is None

# premier return rencontré = sortie immédiate
def signe(x):
    if x > 0:
        return 1
    return -1
assert signe(3) == 1 and signe(-3) == -1

# la variable locale n'existe pas à l'extérieur
try:
    res
except NameError:
    pass
else:
    raise AssertionError("res ne devrait pas être globale")


**Pièges.** Oublier `return` (la fonction renvoie `None`) ; croire qu'une variable locale est visible à l'extérieur.

**Astuce.** Une fonction s'arrête au **premier** `return` rencontré.


## 1.4 Compréhensions de listes

**Définition.** Une compréhension construit une liste en une expression : `[expression for x in iterable if condition]`.

**Quand l'utiliser.** Pour des transformations ou filtrages simples et lisibles. Éviter les compréhensions imbriquées difficiles à relire.


In [ ]:
carres_pairs = [k * k for k in range(10) if k % 2 == 0]
assert carres_pairs == [0, 4, 16, 36, 64]

# équivalent avec une boucle
res = []
for k in range(10):
    if k % 2 == 0:
        res.append(k * k)
assert res == carres_pairs

# dict-compréhension
carre_de = {k: k * k for k in range(4)}
assert carre_de == {0: 0, 1: 1, 2: 4, 3: 9}


**Pièges.** Ne pas confondre le `if` de filtrage (après le `for`) et l'expression conditionnelle (avant le `for`).

**Astuce.** `sum(k * k for k in range(10))` évite de créer une liste intermédiaire : c'est un générateur.


## 1.5 Exceptions

**Définition.** Une exception signale une erreur à l'exécution ; `try / except` permet de la traiter proprement.

**Quand l'utiliser.** Pour gérer des cas prévisibles (division par zéro, conversion invalide). Ne pas masquer toutes les erreurs avec un `except` nu.


In [ ]:
try:
    quotient = 1 / 0
except ZeroDivisionError:
    quotient = None
assert quotient is None

# conversion robuste
def entier(txt):
    try:
        return int(txt)
    except ValueError:
        return None

assert entier("42") == 42
assert entier("abc") is None

# finally s'exécute toujours
trace = []
try:
    trace.append("try")
finally:
    trace.append("finally")
assert trace == ["try", "finally"]


**Pièges.** `except:` sans type attrape tout (y compris `KeyboardInterrupt`) ; l'ordre des `except` compte (du plus précis au plus général).

**Astuce.** Préférer `raise ValueError("message")` à un code d'erreur ambigu.


## 1.6 Lecture et écriture de fichiers

**Définition.** `open(chemin, mode)` ouvre un fichier ; le bloc `with` garantit sa fermeture automatique.

**Quand l'utiliser.** Pour lire ou écrire des données persistantes. Toujours préciser le mode (`"r"`, `"w"`, `"a"`).


In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as dossier:
    chemin = os.path.join(dossier, "notes.txt")

    # écriture (mode "w" : écrase)
    with open(chemin, "w", encoding="utf-8") as f:
        f.write("bonjour\n")
        f.write("CPGE\n")

    # lecture intégrale
    with open(chemin, "r", encoding="utf-8") as f:
        contenu = f.read()
    assert contenu == "bonjour\nCPGE\n"

    # lecture ligne par ligne, sans le saut de ligne
    with open(chemin, "r", encoding="utf-8") as f:
        lignes = [ligne.rstrip("\n") for ligne in f]
    assert lignes == ["bonjour", "CPGE"]

    # ajout en fin de fichier (mode "a")
    with open(chemin, "a", encoding="utf-8") as f:
        f.write("fin\n")
    with open(chemin, "r", encoding="utf-8") as f:
        assert f.read().endswith("fin\n")


**Pièges.** Le mode `"w"` **écrase** le fichier ; oublier `encoding="utf-8"` casse les accents ; sans `with`, le fichier peut rester ouvert.

**Astuce.** `with open(...) as f:` ferme le fichier même si une exception survient.


## Récapitulatif

| Notion | À retenir |
|---|---|
| Types | `int` en précision arbitraire, typage dynamique |
| Boucles | `for` = tours connus, `while` = condition d'arrêt |
| Fonctions | `return` obligatoire, portée locale |
| Compréhensions | filtre après `for`, expression avant |
| Exceptions | `try/except/finally`, du plus précis au plus général |
| Fichiers | `with open(..., encoding="utf-8")` |
